##### Module Imports

In [90]:
import pandas as pd
from sklearn.feature_selection import RFE               # For Recursive Feature Elimination
from sklearn.metrics import accuracy_score              # Function to calculate model accuracy
from sklearn.model_selection import train_test_split    # Splitting a dataset into trian and test datasets	
from xgboost import XGBClassifier

import time                                             # Measuring time for training+

import sys                                              # For importing custom modules (from parent dirs)
sys.path.append('..')
from tools import serialTools, captureTools, eval

##### Global Vars

In [91]:
datasetsPath = '../datasets/harus/'

In [92]:
xNames =    []                              # List of feature (column) names
xtrain =    pd.DataFrame()    # Training data: Features
xtest =     pd.DataFrame()    # Testing data: Features
ytrain =    pd.DataFrame(columns=['label']) # Training data: Labels
ytest =     pd.DataFrame(columns=['label']) # Testing data: Labels
bestIter = 0
initialAcc = 0

##### Func: Import Data

In [93]:
def importData():
    global xtrain, xtest
    # Importing names of the features/columns
    print(f'Importing data...')
    with open(datasetsPath + 'UCI HAR Dataset/features.txt') as f:
        for line in f:                          # Reading each line
            parts = line.strip().split(' ')     # Splitting each line by space

            if len(parts) > 1:                  # If the line has more than 1 element
                label = parts[1]                # The second element is the label
                dupe = False                    # Label is not a dupe, yet!
                index = 0                       # Rising index for naming
                amount = xNames.count(label)    # Check, if the label is already in the list
                
                while amount!=0:                # If a label is already in the list, add a number (index) to it 
                    index = index+1
                    newName = label + '_' + str(index)
                    amount = xNames.count(newName)
                    dupe = True
                if dupe == True:                # If the label has been a duplicate, use the new name
                    xNames.append(newName)    
                else:
                    xNames.append(label)        # Otherwise, just use the original name

    xtrain =    pd.DataFrame(columns=xNames)    # Training data: Features

    # Importing xtrain
    with open(datasetsPath + 'UCI HAR Dataset/train/X_train.txt', 'r') as f:
        for line in f:
            liste = line.strip().split(' ')      # Create a list of every object in the list thats seperated by " "
            liste = [i for i in liste if i != ''] # Remove empty strings
            liste = [float(i) for i in liste]     # Cast every object in the list to float
            xtrain.loc[len(xtrain)] = liste          # Add new_list as a new row to the dataframe

    xtest =     pd.DataFrame(columns=xNames)    # Testing data: Features

    # Importing xtest
    with open(datasetsPath + 'UCI HAR Dataset/test/X_test.txt', 'r') as f:
        for line in f:
            liste = line.strip().split(' ')      # Create a list of every object in the list thats seperated by " "
            liste = [i for i in liste if i != ''] # Remove empty strings
            liste = [float(i) for i in liste]     # Cast every object in the list to float
            xtest.loc[len(xtest)] = liste          # Add new_list as a new row to the dataframe

    # Importing ytrain
    with open(datasetsPath + 'UCI HAR Dataset/train/y_train.txt', 'r') as f:
        labels = []
        for line in f:
            labels.append(int(line.strip())-1)
        
    ytrain['label'] = labels

    # Importing ytest
    with open(datasetsPath + 'UCI HAR Dataset/test/y_test.txt', 'r') as f:
        labels = []
        for line in f:
            labels.append(int(line.strip())-1)
        
    ytest['label'] = labels

    # Preprocessing -> Setting dtypes to columns
    for col in xtrain.columns:
        xtrain[col] = xtrain[col].astype('float32')

    for col in ytrain.columns:
        ytrain[col] = ytrain[col].astype('int32')

    print(f'Data imported!')

##### Func: Train Model

In [94]:
def trainModel(model: XGBClassifier, feats: pd.DataFrame, labels: pd.DataFrame, setBestIter: bool = False, evalset: list = None):
    global bestIter
    
    if setBestIter == True:
        model.set_params(
            objective='multi:softmax',
            num_class=6,
            learning_rate=0.1,
            n_estimators=10000,
            early_stopping_rounds=100,
            max_depth=3
        )
        model.fit(
            feats, labels,
            eval_set = evalset,
            verbose = False
        )
        bestIter = model.best_iteration
    else:
        if bestIter == 0:
            print('BestIter = 0 -> Something is wrong!')
        model.set_params(
            objective='multi:softmax',
            num_class=6,
            learning_rate=0.1,
            n_estimators=bestIter,
            early_stopping_rounds=None,
            max_depth=3
        )
        model.fit(feats,labels)

##### Func: Train Base Model

In [95]:
def trainBase():
    print("Training base model...")
    model = XGBClassifier()
    evalset = [(xtrain,ytrain),(xtest,ytest)]
    trainModel(model, xtrain, ytrain, True, evalset)
    trainModel(model, xtrain, ytrain)
    model.save_model("quicksave.json")
    print("Base model trained and saved as quicksave.json!")

##### Func: Get Important Features

In [96]:
def getImportantFeatures(model, treshold = 0.0): # 0: Score, 1: Name
    fis = eval.getFeatureImportances(model)
    importantFeatures = []
    [importantFeatures.append(name[1]) for name in fis if name[0] > treshold]
    return importantFeatures

##### Func: Feature Selection - Treshold 0.0

In [97]:
def featureSelection(xtrain, xtest, ytrain, ytest, max_decrease = 0.0, model1 = None):
    global initialAcc
    print("Running feature selection (Treshold: 0.0)...")
    if model1 == None:
        model1 = XGBClassifier()
        evalset1 = [(xtrain,ytrain),(xtest,ytest)]
        trainModel(model1, xtrain, ytrain, True, evalset1)
        trainModel(model1, xtrain, ytrain)
        initialAcc = accuracy_score(ytest, model1.predict(xtest))
    
    accscore1 = accuracy_score(ytest, model1.predict(xtest)) 
    ifeats = getImportantFeatures(model1)

    xtrain2 = xtrain[ifeats]
    xtest2 = xtest[ifeats]
    model2 = XGBClassifier()
    evalset2 = [(xtrain2,ytrain),(xtest2,ytest)]
    trainModel(model2, xtrain2, ytrain, True, evalset2)
    trainModel(model2, xtrain2, ytrain)
    accscore2 = accuracy_score(ytest, model2.predict(xtest2)) 
    eval.getAccuracy(ytest, model2.predict(xtest2))

    print(f'Model 1 - #Features: {len(model1.feature_names_in_)}\tAccuracy Score: {accscore1}')
    print(f'Model 2 - #Features: {len(model2.feature_names_in_)}\tAccuracy Score: {accscore2}\n')

    if (initialAcc - accscore2)<=max_decrease and len(model1.feature_names_in_) != len(model2.feature_names_in_):
        featureSelection(xtrain2, xtest2, ytrain, ytest, max_decrease, model2)
    else:
        return [model1, model1.feature_names_in_, xtrain, xtest]

##### Func: Feature Selection - Treshold aus 1st Feature Importances<br>
Eignet sich um einen Graphen mit #Features gegen Accuracy zu erstellen

In [98]:
def featureSelection2(model, xtrain, xtest, ytrain, ytest, threshold):
    
    print(f"Running feature selection 2 (Treshold: {threshold})...")
    
    ifeats = getImportantFeatures(model, threshold)

    xtrain2 = xtrain[ifeats]
    xtest2 = xtest[ifeats]
    model2 = XGBClassifier()
    evalset2 = [(xtrain2,ytrain),(xtest2,ytest)]

    trainModel(model2, xtrain2, ytrain, True, evalset2)
    trainModel(model2, xtrain2, ytrain)

    accscore = accuracy_score(ytest, model2.predict(xtest2)) 

    return [accscore, ifeats, xtrain2, xtest2, model2]
    

##### Func: Feature Selection 2 Iter

In [99]:
def runFeatureSelection2(xtrain, ytrain, xtest, ytest):
    model = XGBClassifier()
    model.load_model("quicksave.json")
    tresholds = sorted(list(set(model.feature_importances_)))
    baseAcc = accuracy_score(ytest, model.predict(xtest))
    max_decrease = 5.0

    for tresh in tresholds:
        temp = featureSelection2(
            model, 
            xtrain,xtest,ytrain,ytest,
            tresh
        )
        print(f'Threshold: {tresh} \nAccuracy: {temp[0]} \tFeatures: {len(temp[1])}\n')
        if baseAcc - temp[0] > max_decrease:
            break

##### Func: Feature Selection Iter 20th

In [100]:
# runFeatureSelection2 jeder 20. Grenzwert
def runFeatureSelection2Nth(xtrain, xtest, ytrain, ytest, n):
    model = XGBClassifier()
    model.load_model("quicksave.json")
    tresholds = sorted(list(set(model.feature_importances_)))
    baseAcc = accuracy_score(ytest, model.predict(xtest))
    max_decrease = 5.0

    for tresh in tresholds[:n]:
        temp = featureSelection2(
            model, 
            xtrain,xtest,ytrain,ytest,
            tresh
        )
        print(f'Threshold: {tresh} \nAccuracy: {temp[0]} \tFeatures: {len(temp[1])}\n')
        if baseAcc - temp[0] > max_decrease:
            break
    

##### Func: Feature Selection - RFE
- [XGBoost Feature Selection with RFE](https://xgboosting.com/xgboost-feature-selection-with-rfe/)
- [Recursive Feature Elimination (RFE) for Feature Selection in Python](https://machinelearningmastery.com/rfe-feature-selection-in-python/)
- [Scikit Learn: RFE](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html)

In [101]:

def featureSelectionRfe(xtrain, xtest, ytrain, ytest, compareTop20: bool = False): # Might take up to 30 minutes
    if compareTop20 == False:
        print("Training model using RFE...")
    else:
        print("Training model using RFE and comparing to top 20 features approach...")
    # Initialize XGBClassifier and RFE Wrapper
    start_time_get_rfe = time.perf_counter()
    modelForRfe = XGBClassifier()
    rfe = RFE(estimator=modelForRfe, n_features_to_select=20)

    # Fit the RFE Wrapper to the training data
    rfe.fit(xtrain, ytrain)

    # Train a model using the features selected by RFE
    start_time = time.perf_counter()
    modelRfe = XGBClassifier()
    modelRfe.fit(xtrain.iloc[:, rfe.get_support()], ytrain)
    end_time = time.perf_counter()
    rfe_time  = end_time - start_time
    rfe_time_get_rfe = end_time - start_time_get_rfe
    
    # Make predictions using the RFE model
    yhatRfe = modelRfe.predict(xtest.iloc[:, rfe.support_])
    
    # Print metrics
    print(f'Accuracy using RFE: {accuracy_score(ytest, yhatRfe)}')
    print(f'Training time using RFE: {rfe_time}')
    print(f'Training time using RFE (including getting selected features): {rfe_time}')

    if compareTop20:
        # Get top 20 feature importances
        start_time_get_importances = time.perf_counter()
        model = XGBClassifier()
        model.fit(xtrain, ytrain)
        importance_scores = model.feature_importances_
        selected_features = importance_scores.argsort()[-20:]

        # Train a model using the top 20 features by importance scores
        start_time = time.perf_counter()
        modelTop20 = XGBClassifier()
        modelTop20.fit(xtrain.iloc[:, selected_features], ytrain)
        stop_time = time.perf_counter()
        modelTop20_get_importances_time = stop_time - start_time_get_importances
        modelTop20_time  = stop_time - start_time

        # Make predictions using both models        
        yhatTop20 = modelTop20.predict(xtest.iloc[:, selected_features])

        # Compare accuracy and training time
        print(f'Accuracy using Top 20: {accuracy_score(ytest, yhatTop20)}')
        print(f'Training time using Top20 (without getting importance scores): {modelTop20_time}')
        print(f'Training time using Top20 (including getting importance scores): {modelTop20_get_importances_time}')

##### Main:

In [102]:
# importData()
# trainBase()
# featureSelection(xtrain,xtest,ytrain,ytest,10.0)
# runFeatureSelection2(xtrain,xtest,ytrain,ytest)
# runFeatureSelection2Nth(xtrain,xtest,ytrain,ytest,20)
# featureSelectionRfe(xtrain,xtest,ytrain,ytest, compareTop20=True)